# AniMer dense-anchor point-to-plane articulation: `hippo_05`

This notebook runs `articulate/config/dense_anchor_point_to_plane_animer.yaml` from the raw `hippo_05` scan through fresh AniMer inference, robust joint and dense-anchor triangulation, sparse fitting, point-to-plane ICP, Chamfer polishing, confidence-weighted fusion, and production output.

The main goal is to decide whether the triangulated joints and anchors are trustworthy **before** optimization. It therefore exposes per-view 2D observations, triangulated reprojections, robust confidence, view support, covariance, 3D placement, and raw-surface grounding. Run it in the dedicated `AniMer` Conda environment with CUDA. No saved fit is used as an input.

## 1. Repository, imports, and exact Hydra preset

In [ ]:
import os
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")

import sys
from pathlib import Path


def find_project_root(start=Path.cwd()):
    for candidate in (Path(start).resolve(), *Path(start).resolve().parents):
        if (candidate / "articulate/config/dense_anchor_point_to_plane_animer.yaml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the atm-dev repository root.")


PROJECT_ROOT = find_project_root()
ANIMER_ROOT = PROJECT_ROOT / "pkg/AniMer"
os.chdir(PROJECT_ROOT)
for path in (PROJECT_ROOT, ANIMER_ROOT):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print(f"Repository: {PROJECT_ROOT}")

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
import torch
import trimesh
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from omegaconf import OmegaConf

from articulate.shape_refiner.dense_anchor_mhr_refiner import triangulation_covariances
from articulate.shape_refiner.dense_anchor_point_to_plane_smal_refiner import (
    DenseAnchorPointToPlaneSMALShapeRefiner,
)
from articulate.shape_refiner.dense_anchor_smal_refiner import (
    closest_points_on_mesh,
    project_smal_anchor_vertices,
)
from articulate.shape_refiner.mhr_refiner import reprojection_errors_all
from articulate.shape_refiner.smal_refiner import project_points

PYVISTA_BACKEND = "static"
pv.set_jupyter_backend(PYVISTA_BACKEND)
pv.global_theme.background = "white"
pv.global_theme.font.color = "black"

In [ ]:
SHAPE_NAME = "hippo_05"
RAW_DIR = Path("/home/knpob/work/data/shape-corr/SMAL_r/off")
RAW_PATH = RAW_DIR / f"{SHAPE_NAME}.off"
NOTEBOOK_OUT_DIR = PROJECT_ROOT / "output/notebook_smal_animer_dense_anchor_point_to_plane_hippo05_e2e"
DEVICE = "cuda"

overrides = [
    f"shape_dir={RAW_DIR}",
    f"out_dir={NOTEBOOK_OUT_DIR}",
    f"shape_loader.filename_filter={SHAPE_NAME}",
    f"device={DEVICE}",
]
if GlobalHydra.instance().is_initialized():
    GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "articulate/config"), version_base=None):
    cfg = compose(config_name="dense_anchor_point_to_plane_animer", overrides=overrides)

required = [RAW_PATH, PROJECT_ROOT / cfg.shape_estimator.checkpoint_path, ANIMER_ROOT]
missing = [str(path) for path in required if not path.exists()]
assert not missing, "Missing required assets: " + ", ".join(missing)
if str(cfg.device).startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("Run this notebook in the AniMer Conda environment on a CUDA machine.")

assert cfg.shape_refiner._target_.endswith("DenseAnchorPointToPlaneSMALShapeRefiner")
assert cfg.shape_refiner.n_iters_keypoints == 1000
assert cfg.shape_refiner.n_iters_anchors == 1000
assert cfg.shape_refiner.n_iters_icp == 500
assert cfg.shape_refiner.icp_correspondence_interval == 10
assert cfg.shape_refiner.n_iters_surface == 100
assert cfg.shape_refiner.n_views_surface == 0
assert not cfg.shape_refiner.early_stop_chamfer_plateau
assert not cfg.shape_refiner.early_stop_validation_regression
print(OmegaConf.to_yaml(cfg, resolve=True))

In [ ]:
def as_polydata(vertices, faces):
    vertices = np.asarray(vertices)
    faces = np.asarray(faces, dtype=np.int64)
    return pv.PolyData(vertices, np.column_stack([np.full(len(faces), 3), faces]).ravel())


def set_parallel_view(plotter, view):
    getattr(plotter, f"view_{view}")()
    plotter.enable_parallel_projection()


def line_polydata(starts, ends):
    starts, ends = np.asarray(starts), np.asarray(ends)
    points = np.concatenate([starts, ends], axis=0)
    count = len(starts)
    lines = np.column_stack([np.full(count, 2), np.arange(count), np.arange(count) + count]).ravel()
    result = pv.PolyData(points)
    result.lines = lines
    return result


def rgba_overlay(rgb, rgba):
    rgb = np.asarray(rgb, dtype=np.float32) / 255.0
    rgba = np.asarray(rgba, dtype=np.float32) / 255.0
    alpha = rgba[..., 3:4]
    rendered = rgba[..., :3] * alpha + rgb * (1.0 - alpha)
    return np.clip(0.55 * rgb + 0.45 * rendered, 0.0, 1.0)


def robust_median(values, mask, axis=0):
    masked = np.where(mask, values, np.nan)
    return np.nanmedian(masked, axis=axis)

## 2. Notebook-only stage tracing

This thin subclass delegates every optimization step to production. It only retains detached meshes at the joint, sparse-anchor, ICP, and Chamfer boundaries plus per-refresh ICP support counts.

In [ ]:
class TracingDenseAnchorPointToPlaneAniMerRefiner(DenseAnchorPointToPlaneSMALShapeRefiner):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.stage_vertices = {}
        self.icp_boundary_vertices = []
        self.icp_boundary_chamfers = []
        self.icp_refresh_history = []

    def _capture(self, name, results, shape_prior, parameters):
        betas, global_orient, pose = parameters
        rotation, translation, scale = self.stack_alignment_params(results, dtype=betas.dtype)
        with torch.no_grad():
            vertices, joints = shape_prior.generate(
                betas, global_orient, pose, rotation, scale, translation
            )
        self.stage_vertices[name] = vertices.detach().cpu().numpy()
        self.stage_vertices[f"{name}_joints"] = joints.detach().cpu().numpy()

    def _sparse_anchor_optim(self, results, shape_prior, betas, global_orient, pose):
        self._capture("post_joint", results, shape_prior, (betas, global_orient, pose))
        optimized = super()._sparse_anchor_optim(results, shape_prior, betas, global_orient, pose)
        self._capture("post_sparse", results, shape_prior, optimized)
        return optimized

    def _validation_chamfer_for_parameters(
        self, generate_vertices, model_indices, raw_points, probes, faces, canonical_vertices
    ):
        with torch.no_grad():
            self.icp_boundary_vertices.append(generate_vertices().detach().cpu().numpy())
        value = super()._validation_chamfer_for_parameters(
            generate_vertices, model_indices, raw_points, probes, faces, canonical_vertices
        )
        self.icp_boundary_chamfers.append(float(value))
        return value

    def _refresh_icp_correspondences(
        self, vertices, faces, source_indices, target_points, target_normals, tree, maximum_distance
    ):
        correspondences, fallback = super()._refresh_icp_correspondences(
            vertices, faces, source_indices, target_points, target_normals, tree, maximum_distance
        )
        self.icp_refresh_history.append(correspondences.accepted_per_view.copy())
        return correspondences, fallback

    def _dense_optim(self, results, mesh, shape_prior, betas, global_orient, pose):
        optimized = super()._dense_optim(results, mesh, shape_prior, betas, global_orient, pose)
        self._capture("post_chamfer", results, shape_prior, optimized)
        return optimized

## 3. Initialize components and render `hippo_05`

In [ ]:
shape_loader = instantiate(cfg.shape_loader)
view_sampler = instantiate(cfg.view_sampler)
shape_estimator = instantiate(cfg.shape_estimator)
shape_prior = instantiate(cfg.shape_prior)
diagnostic_refiner = instantiate(cfg.shape_refiner)
refiner_kwargs = OmegaConf.to_container(cfg.shape_refiner, resolve=True)
refiner_kwargs.pop("_target_")
shape_refiner = TracingDenseAnchorPointToPlaneAniMerRefiner(**refiner_kwargs)
output_handler = instantiate(cfg.output_handler)

started = time.perf_counter()
shape_estimator.init_estimator()
shape_prior.load()
shape_prior.load_from_estimator(shape_estimator)
print(f"Initialized AniMer and checkpoint-owned SMAL in {time.perf_counter() - started:.1f} s")
shape_files = shape_loader.find_shape_files()
assert shape_files == [str(RAW_PATH)], shape_files

In [ ]:
shape = shape_loader.pipeline(shape_files[0])
raw_mesh = shape["mesh"]
camera_views = view_sampler.sample_views()
view_sampling = view_sampler.pipeline(shape=shape, camera_views=camera_views)
assert len(camera_views) == len(view_sampling["observations"]) == 18
print(f"Raw mesh: {len(raw_mesh.vertices):,} vertices / {len(raw_mesh.faces):,} faces")

fig, axes = plt.subplots(3, 6, figsize=(16, 8), constrained_layout=True)
for axis, (view_key, observation) in zip(axes.flat, view_sampling["observations"].items()):
    axis.imshow(observation["rgb"])
    axis.set_title(f"az={view_key[0]:g}, el={view_key[1]:g}", fontsize=8)
    axis.axis("off")
fig.suptitle("Production AniMer camera observations")
plt.show()

## 4. Fresh AniMer inference and similarity alignment

In [ ]:
frontend_started = time.perf_counter()
shape_estimation = shape_estimator.pipeline(view_sampling=view_sampling)
frontend_seconds = time.perf_counter() - frontend_started
results = shape_estimation["results"]
view_keys = list(results)
assert len(results) >= 3

print(f"Accepted AniMer views: {len(results)} / 18 in {frontend_seconds:.1f} s")
print("view             IoU       RMSE      inliers   ratio")
for view_key, result in results.items():
    diagnostic = result["confidence"]
    print(
        f"{str(view_key):16s} {float(result['iou']):.4f}   "
        f"{float(diagnostic['alignment_rmse']):.6f}   "
        f"{int(diagnostic['num_inliers']):7d}   {float(diagnostic['inlier_ratio']):.4f}"
    )

In [ ]:
panel_ids = np.linspace(0, len(view_keys) - 1, min(6, len(view_keys)), dtype=int)
fig, axes = plt.subplots(2, len(panel_ids), figsize=(3 * len(panel_ids), 6), constrained_layout=True)
for column, view_index in enumerate(panel_ids):
    key = view_keys[view_index]
    result = results[key]
    axes[0, column].imshow(view_sampling["observations"][key]["rgb"])
    axes[1, column].imshow(rgba_overlay(view_sampling["observations"][key]["rgb"], result["img_rgba_smal"]))
    axes[0, column].set_title(f"input {key}", fontsize=8)
    axes[1, column].set_title(f"AniMer IoU={result['iou']:.3f}", fontsize=8)
    axes[0, column].axis("off")
    axes[1, column].axis("off")
plt.show()

## 5. Joint triangulation: observations, reprojection, support, and covariance

AniMer predicts crop-normalized 2D joints. Production converts them to full-image coordinates and flips the image y-axis before robust DLT triangulation. For display below, both observations and triangulated reprojections are flipped back to top-left image coordinates. Green observations and cyan reprojections should coincide on the same anatomical landmark.

In [ ]:
camera_configs = shape_estimation["camera_configs"]
triangulated_joints, joint_used_views, joint_confidence = diagnostic_refiner.joint_triangulate(
    results, camera_configs=camera_configs
)
joint_observations = diagnostic_refiner._joint_observations(results, camera_configs)
projection_matrices = diagnostic_refiner._projection_data(results)
joint_errors, joint_depths = reprojection_errors_all(
    projection_matrices, triangulated_joints, joint_observations
)
joint_covariances, joint_support = triangulation_covariances(
    projection_matrices,
    triangulated_joints,
    joint_observations,
    joint_confidence,
    confidence_threshold=diagnostic_refiner.conf_thresh,
    pixel_sigma=diagnostic_refiner.joint_pixel_sigma,
    damping=diagnostic_refiner.covariance_damping,
    uncertainty_floor=diagnostic_refiner.covariance_floor,
)
joint_valid = diagnostic_refiner._joint_valid
expanded_joint_valid = (
    np.all(np.isfinite(triangulated_joints), axis=1)
    & np.all(np.isfinite(joint_covariances), axis=(1, 2))
    & (joint_support >= diagnostic_refiner.min_joint_views)
)
np.testing.assert_array_equal(joint_valid, expanded_joint_valid)
joint_inliers = (joint_confidence >= diagnostic_refiner.conf_thresh) & np.isfinite(joint_errors)
joint_median_error = robust_median(joint_errors, joint_inliers, axis=0)
joint_position_sigma = np.sqrt(np.maximum(np.trace(joint_covariances, axis1=1, axis2=2), 0.0))

print(f"Valid triangulated joints: {joint_valid.sum()} / {len(joint_valid)}")
print(f"Support range: {joint_support.min()}–{joint_support.max()} views")
print(f"Median accepted reprojection error: {np.nanmedian(joint_median_error):.3f} px")
worst_joint_ids = np.argsort(np.nan_to_num(joint_median_error, nan=np.inf))[-6:][::-1]
print("\nJoints requiring closest visual inspection:")
print("joint  valid  support  median_reproj_px  position_sigma")
for joint_id in worst_joint_ids:
    print(f"{joint_id:5d}  {str(bool(joint_valid[joint_id])):5s}  {joint_support[joint_id]:7d}  {joint_median_error[joint_id]:16.3f}  {joint_position_sigma[joint_id]:14.6f}")

In [ ]:
image_height = int(camera_configs["h"])
panel_ids = np.linspace(0, len(view_keys) - 1, min(6, len(view_keys)), dtype=int)
fig, axes = plt.subplots(2, 3, figsize=(15, 10), constrained_layout=True)
for axis, view_index in zip(axes.flat, panel_ids):
    key = view_keys[view_index]
    projected, depth = project_points(projection_matrices[view_index], triangulated_joints)
    observed_display = joint_observations[view_index].copy()
    projected_display = projected.copy()
    observed_display[:, 1] = image_height - observed_display[:, 1]
    projected_display[:, 1] = image_height - projected_display[:, 1]
    accepted = joint_inliers[view_index] & (depth > 0)
    axis.imshow(view_sampling["observations"][key]["rgb"])
    axis.scatter(observed_display[accepted, 0], observed_display[accepted, 1], s=30, c="lime", label="AniMer observation")
    axis.scatter(projected_display[accepted, 0], projected_display[accepted, 1], s=20, c="cyan", marker="x", label="triangulated reprojection")
    for start, end in zip(observed_display[accepted], projected_display[accepted]):
        axis.plot([start[0], end[0]], [start[1], end[1]], color="yellow", linewidth=0.8, alpha=0.8)
    axis.set_title(f"{key}: {accepted.sum()} joint inliers")
    axis.set_xlim(0, camera_configs["w"])
    axis.set_ylim(camera_configs["h"], 0)
    axis.axis("off")
axes.flat[0].legend(fontsize=8, loc="lower left")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), constrained_layout=True)
axes[0].bar(np.arange(len(joint_support)), joint_support, color=np.where(joint_valid, "#31a354", "#de2d26"))
axes[0].axhline(diagnostic_refiner.min_joint_views, color="black", linestyle="--")
axes[0].set(title="Views supporting each triangulated joint", ylabel="views")
image = axes[1].imshow(joint_confidence, aspect="auto", cmap="magma", vmin=0, vmax=1)
axes[1].set(title="Tukey confidence after reprojection gating", ylabel="view index")
fig.colorbar(image, ax=axes[1], label="confidence")
axes[2].bar(np.arange(len(joint_median_error)), joint_median_error, color="#3182bd", label="median reprojection error (px)")
axes[2].plot(np.arange(len(joint_position_sigma)), joint_position_sigma, color="#de2d26", marker="o", label="sqrt(trace(covariance)) (world units)")
axes[2].set(title="Joint reprojection error and 3D uncertainty", xlabel="SMAL joint ID")
axes[2].legend()
plt.show()

In [ ]:
betas_initial, global_orient_initial, pose_initial = diagnostic_refiner.stack_smal_params(results)
rotation_initial, translation_initial, scale_initial = diagnostic_refiner.stack_alignment_params(
    results, dtype=betas_initial.dtype
)
with torch.no_grad():
    _, per_view_initial_joints = shape_prior.generate(
        betas_initial, global_orient_initial, pose_initial,
        rotation_initial, scale_initial, translation_initial,
    )
per_view_initial_joints = per_view_initial_joints.detach().cpu().numpy()
joint_disagreement = np.linalg.norm(per_view_initial_joints - triangulated_joints[None], axis=2)

bone_pairs = [pair for pair in shape_prior.get_skeleton() if max(pair) < len(triangulated_joints)]
finite_bones = [pair for pair in bone_pairs if np.isfinite(triangulated_joints[list(pair)]).all()]
for view in ("xy", "yz", "zx"):
    plotter = pv.Plotter()
    plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#d9d9d9", opacity=0.30, smooth_shading=True)
    plotter.add_points(triangulated_joints[joint_valid], scalars=joint_position_sigma[joint_valid], cmap="magma", point_size=14, render_points_as_spheres=True, scalar_bar_args={"title": "joint 3D sigma"})
    if finite_bones:
        starts = np.stack([triangulated_joints[a] for a, _ in finite_bones])
        ends = np.stack([triangulated_joints[b] for _, b in finite_bones])
        plotter.add_mesh(line_polydata(starts, ends), color="#2c7fb8", line_width=4)
    plotter.add_title(f"Triangulated skeleton inside raw hippo_05 — {view}")
    set_parallel_view(plotter, view)
    plotter.show(jupyter_backend=PYVISTA_BACKEND)

plt.figure(figsize=(12, 4))
plt.boxplot(joint_disagreement, positions=np.arange(len(triangulated_joints)), showfliers=False)
plt.xlabel("SMAL joint ID")
plt.ylabel("aligned AniMer joint distance to triangulation")
plt.title("Per-view 3D joint disagreement before joint optimization")
plt.show()

## 6. Dense-anchor triangulation: visibility, reprojection, support, and covariance

The anchors use the shared 1,024-vertex geodesic-FPS template. A projected anchor is accepted only when it is in frame, visible in the rendered AniMer depth, and safely inside the raw silhouette. Robust triangulation then requires at least three supported views.

In [ ]:
canonical_vertices = diagnostic_refiner._neutral_template(
    shape_prior, betas_initial, global_orient_initial, pose_initial
)
faces = np.asarray(shape_prior.faces, dtype=np.int64)
anchor_indices = diagnostic_refiner._load_or_create_anchors(canonical_vertices, faces)
diagnostic_refiner._canonical_vertices = canonical_vertices
diagnostic_refiner._anchor_indices = anchor_indices
anchor_observations, anchor_used_views, anchor_confidence = diagnostic_refiner._triangulate_anchors(
    results, raw_mesh
)
triangulated_anchors = diagnostic_refiner._anchor_targets
anchor_valid = diagnostic_refiner._anchor_valid
anchor_uv = anchor_observations.copy()
anchor_uv[:, :, 1] = camera_configs["h"] - anchor_uv[:, :, 1]
anchor_errors, anchor_depths = reprojection_errors_all(
    projection_matrices, triangulated_anchors, anchor_uv
)
anchor_covariances, anchor_support = triangulation_covariances(
    projection_matrices, triangulated_anchors, anchor_uv, anchor_confidence,
    confidence_threshold=diagnostic_refiner.conf_thresh,
    pixel_sigma=diagnostic_refiner.anchor_pixel_sigma,
    damping=diagnostic_refiner.covariance_damping,
    uncertainty_floor=diagnostic_refiner.covariance_floor,
)
expanded_anchor_valid = (
    np.all(np.isfinite(triangulated_anchors), axis=1)
    & np.all(np.isfinite(anchor_covariances), axis=(1, 2))
    & (anchor_support >= diagnostic_refiner.min_anchor_views)
)
np.testing.assert_array_equal(anchor_valid, expanded_anchor_valid)
anchor_inliers = (anchor_confidence >= diagnostic_refiner.conf_thresh) & np.isfinite(anchor_errors)
anchor_median_error = robust_median(anchor_errors, anchor_inliers, axis=0)
anchor_position_sigma = np.sqrt(np.maximum(np.trace(anchor_covariances, axis1=1, axis2=2), 0.0))
print(f"Accepted triangulated anchors: {anchor_valid.sum()} / {len(anchor_valid)}")
print(f"Median support among accepted anchors: {np.median(anchor_support[anchor_valid]):g} views")
print(f"Median accepted reprojection error: {np.nanmedian(anchor_median_error[anchor_valid]):.3f} px")
accepted_ids = np.flatnonzero(anchor_valid)
worst_anchor_ids = accepted_ids[np.argsort(anchor_median_error[accepted_ids])[-10:][::-1]]
print("\nAccepted anchors with the largest median reprojection errors:")
print("anchor  vertex  support  median_reproj_px  position_sigma")
for anchor_id in worst_anchor_ids:
    print(f"{anchor_id:6d}  {anchor_indices[anchor_id]:6d}  {anchor_support[anchor_id]:7d}  {anchor_median_error[anchor_id]:16.3f}  {anchor_position_sigma[anchor_id]:14.6f}")

In [ ]:
projected_candidates = []
for result in results.values():
    predicted = result["amr_output"]["pred_vertices"][0].detach().cpu().numpy()
    camera = result["cam_smal"]
    pixels, _ = project_smal_anchor_vertices(
        predicted[anchor_indices], camera["pred_cam_t_full"],
        camera["focal_length"], camera["intrinsics"]["camera_center"],
    )
    projected_candidates.append(pixels)
projected_candidates = np.stack(projected_candidates)

panel_ids = np.linspace(0, len(view_keys) - 1, min(4, len(view_keys)), dtype=int)
fig, axes = plt.subplots(2, 2, figsize=(11, 11), constrained_layout=True)
for axis, view_index in zip(axes.flat, panel_ids):
    key = view_keys[view_index]
    usable = np.isfinite(anchor_observations[view_index]).all(axis=1)
    reprojected, depth = project_points(projection_matrices[view_index], triangulated_anchors)
    reprojected[:, 1] = camera_configs["h"] - reprojected[:, 1]
    accepted = anchor_inliers[view_index] & anchor_valid & (depth > 0)
    axis.imshow(view_sampling["observations"][key]["rgb"])
    axis.scatter(projected_candidates[view_index, :, 0], projected_candidates[view_index, :, 1], s=3, c="orange", alpha=0.15, label="all projected")
    axis.scatter(anchor_observations[view_index, usable, 0], anchor_observations[view_index, usable, 1], s=6, c="lime", alpha=0.65, label="visibility-usable")
    axis.scatter(reprojected[accepted, 0], reprojected[accepted, 1], s=7, c="cyan", marker="x", label="3D reprojection")
    axis.set_title(f"{key}: {usable.sum()} usable, {accepted.sum()} robust inliers")
    axis.set_xlim(0, camera_configs["w"])
    axis.set_ylim(camera_configs["h"], 0)
    axis.axis("off")
axes.flat[0].legend(fontsize=7, loc="lower left")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
axes[0, 0].hist(anchor_support[anchor_valid], bins=np.arange(anchor_support.max() + 2) - 0.5, color="#31a354")
axes[0, 0].set(title="Accepted-anchor view support", xlabel="views")
axes[0, 1].hist(anchor_median_error[anchor_valid], bins=50, color="#3182bd")
axes[0, 1].set(title="Accepted-anchor median reprojection error", xlabel="pixels")
axes[1, 0].hist(anchor_position_sigma[anchor_valid], bins=50, color="#756bb1")
axes[1, 0].set(title="Accepted-anchor 3D covariance scale", xlabel="sqrt(trace(covariance))")
axes[1, 1].scatter(anchor_median_error[anchor_valid], anchor_position_sigma[anchor_valid], s=8, alpha=0.45)
axes[1, 1].set(title="2D consistency versus 3D uncertainty", xlabel="median reprojection error (px)", ylabel="3D covariance scale")
plt.show()

rejected = np.all(np.isfinite(triangulated_anchors), axis=1) & ~anchor_valid
plotter = pv.Plotter(shape=(1, 2), window_size=(1500, 650))
plotter.subplot(0, 0)
plotter.add_mesh(as_polydata(canonical_vertices, faces), color="#b3cde3", opacity=0.55, smooth_shading=True)
plotter.add_points(canonical_vertices[anchor_indices], color="#d7301f", point_size=5, render_points_as_spheres=True)
plotter.add_title("1,024 canonical anchor candidates")
plotter.subplot(0, 1)
plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#d9d9d9", opacity=0.30, smooth_shading=True)
if np.any(rejected):
    plotter.add_points(triangulated_anchors[rejected], color="#de2d26", point_size=6, render_points_as_spheres=True, label="rejected")
plotter.add_points(triangulated_anchors[anchor_valid], scalars=anchor_position_sigma[anchor_valid], cmap="viridis", point_size=7, render_points_as_spheres=True, label="accepted")
plotter.add_legend()
plotter.add_title("Triangulated anchors; accepted points colored by covariance")
plotter.show(jupyter_backend=PYVISTA_BACKEND)

for view in ("xy", "yz", "zx"):
    plotter = pv.Plotter()
    plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#d9d9d9", opacity=0.28, smooth_shading=True)
    if np.any(rejected):
        plotter.add_points(triangulated_anchors[rejected], color="#de2d26", point_size=5, render_points_as_spheres=True, label="rejected")
    plotter.add_points(triangulated_anchors[anchor_valid], scalars=anchor_median_error[anchor_valid], cmap="viridis", point_size=7, render_points_as_spheres=True, label="accepted", scalar_bar_args={"title": "median reprojection error (px)"})
    plotter.add_legend()
    plotter.add_title(f"Triangulated anchor placement — {view}")
    set_parallel_view(plotter, view)
    plotter.show(jupyter_backend=PYVISTA_BACKEND)

## 7. Run production refinement and verify diagnostic equivalence

In [ ]:
refiner_started = time.perf_counter()
estimation_refined = shape_refiner.pipeline(shape_estimation=shape_estimation, shape_prior=shape_prior)
refiner_seconds = time.perf_counter() - refiner_started
output_handler.pipeline(estimation_refined=estimation_refined)
articulated_mesh = estimation_refined["mesh_merged"]
confidence = estimation_refined["confidence"]

np.testing.assert_allclose(shape_refiner._last_joint_target, triangulated_joints, rtol=1e-7, atol=1e-8, equal_nan=True)
np.testing.assert_array_equal(shape_refiner._joint_valid, joint_valid)
np.testing.assert_allclose(shape_refiner._anchor_targets, triangulated_anchors, rtol=1e-7, atol=1e-8, equal_nan=True)
np.testing.assert_array_equal(shape_refiner._anchor_valid, anchor_valid)
assert len(shape_refiner.icp_refresh_history) == 50
for suffix in ("obj", "npy", "confidence.npz"):
    assert (output_handler.out_dir / "smal" / f"{SHAPE_NAME}.{suffix}").is_file()

summary = {
    "accepted_views": len(results),
    "valid_joints": int(joint_valid.sum()),
    "accepted_anchors": int(np.asarray(confidence["accepted_anchor_count"]).item()),
    "grounded_anchors": int(np.asarray(confidence["grounded_anchor_count"]).item()),
    "icp_pre_chamfer": float(np.asarray(confidence["icp_pre_validation_chamfer"]).item()),
    "icp_post_chamfer": float(np.asarray(confidence["icp_post_validation_chamfer"]).item()),
    "icp_handoff": str(np.asarray(confidence["icp_chamfer_handoff"]).item()),
    "dense_best_score": float(np.asarray(confidence["dense_best_score"]).item()),
    "refiner_seconds": refiner_seconds,
}
summary

In [ ]:
ground_targets, ground_normals, grounded_mask, body_diagonal = shape_refiner._grounding
ground_ids = np.flatnonzero(grounded_mask)
if len(ground_ids) > 350:
    ground_ids = ground_ids[np.linspace(0, len(ground_ids) - 1, 350, dtype=int)]
plotter = pv.Plotter()
plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#d9d9d9", opacity=0.32, smooth_shading=True)
plotter.add_points(shape_refiner._anchor_targets[shape_refiner._anchor_valid], color="#f16913", point_size=6, render_points_as_spheres=True, label="triangulated anchors")
if len(ground_ids):
    plotter.add_mesh(line_polydata(shape_refiner._anchor_targets[ground_ids], ground_targets[ground_ids]), color="#756bb1", line_width=1, label="accepted grounding")
plotter.add_legend()
plotter.add_title("Triangulated anchors and raw-surface grounding used by sparse fitting")
set_parallel_view(plotter, "yz")
plotter.show(jupyter_backend=PYVISTA_BACKEND)

## 8. Did good triangulation lead to a good fit?

In [ ]:
stage_vertices = {
    "post joint": shape_refiner.stage_vertices["post_joint"],
    "post sparse anchors": shape_refiner.stage_vertices["post_sparse"],
    "pre ICP": shape_refiner.icp_boundary_vertices[0],
    "post ICP": shape_refiner.icp_boundary_vertices[1],
    "post Chamfer": shape_refiner.stage_vertices["post_chamfer"],
}
for name, vertices_by_view in stage_vertices.items():
    plotter = pv.Plotter()
    plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#bdbdbd", opacity=0.30, smooth_shading=True, label="raw")
    plotter.add_mesh(as_polydata(vertices_by_view.mean(axis=0), faces), color="#d7301f", opacity=0.65, smooth_shading=True, label=name)
    plotter.add_legend()
    plotter.add_title(f"Mean per-view surface: {name}")
    set_parallel_view(plotter, "yz")
    plotter.show(jupyter_backend=PYVISTA_BACKEND)

In [ ]:
refresh_history = np.stack(shape_refiner.icp_refresh_history)
steps = np.arange(len(refresh_history)) * cfg.shape_refiner.icp_correspondence_interval
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for view_index, key in enumerate(view_keys):
    axes[0].plot(steps, refresh_history[:, view_index], alpha=0.4, linewidth=1)
axes[0].plot(steps, refresh_history.mean(axis=1), color="black", linewidth=3, label="mean")
axes[0].set(title="ICP correspondence support", xlabel="step", ylabel="accepted matches")
axes[0].legend()
metric_names = ["pre ICP", "post ICP", "post Chamfer"]
metric_values = [
    float(np.asarray(confidence["icp_pre_validation_chamfer"]).item()),
    float(np.asarray(confidence["icp_post_validation_chamfer"]).item()),
    float(np.asarray(confidence["dense_best_score"]).item()),
]
axes[1].bar(metric_names, metric_values, color=["#9ecae1", "#3182bd", "#de2d26"])
axes[1].set(title="Fixed validation Chamfer", ylabel="distance")
plt.show()

In [ ]:
raw_diagonal = max(float(np.linalg.norm(np.ptp(raw_mesh.bounds, axis=0))), 1e-12)
_, vertex_gap, _ = closest_points_on_mesh(raw_mesh, articulated_mesh.vertices)
distance_mesh = as_polydata(articulated_mesh.vertices, articulated_mesh.faces)
distance_mesh["nearest raw gap / diagonal"] = vertex_gap / raw_diagonal
uncertainty_mesh = as_polydata(articulated_mesh.vertices, articulated_mesh.faces)
uncertainty_mesh["fusion uncertainty"] = np.asarray(confidence["vertex_uncertainty"])

for view in ("xy", "yz", "zx"):
    plotter = pv.Plotter()
    plotter.add_mesh(as_polydata(raw_mesh.vertices, raw_mesh.faces), color="#bdbdbd", opacity=0.35, smooth_shading=True, label="raw hippo_05")
    plotter.add_mesh(as_polydata(articulated_mesh.vertices, articulated_mesh.faces), color="#d7301f", opacity=0.65, smooth_shading=True, label="AniMer point-to-plane result")
    plotter.add_legend()
    plotter.add_title(f"Raw versus final articulation — {view}")
    set_parallel_view(plotter, view)
    plotter.show(jupyter_backend=PYVISTA_BACKEND)

for title, mesh, scalar, cmap in (
    ("Final surface error", distance_mesh, "nearest raw gap / diagonal", "magma"),
    ("Fusion uncertainty", uncertainty_mesh, "fusion uncertainty", "inferno"),
):
    plotter = pv.Plotter()
    plotter.add_mesh(mesh, scalars=scalar, cmap=cmap, smooth_shading=True)
    plotter.add_title(title)
    set_parallel_view(plotter, "yz")
    plotter.show(jupyter_backend=PYVISTA_BACKEND)

## Visual correctness checklist

- The green AniMer joint observations and cyan triangulated reprojections must land on the same anatomical landmarks. A consistent vertical offset indicates a crop/image y-convention error.
- The 26-joint skeleton must lie inside the raw hippo without mirrored or swapped legs. Inspect both side and front views; a plausible side view can hide left/right swaps.
- Joint support, reprojection error, covariance, and per-view 3D disagreement should agree. High support alone is not evidence of correct semantics.
- Anchor candidates should be rejected at silhouettes and occlusions, not across broad visible regions. Cyan 3D reprojections should return to the same surface features as green observations.
- Accepted anchors with low 2D error but high 3D covariance indicate weak camera geometry; low values for both can still be consistently attached to the wrong semantic part.
- The notebook assertions verify that the expanded joint and anchor targets and validity masks exactly match those consumed by production.
- Compare post-joint and post-sparse meshes. If geometry first degrades during sparse fitting despite good triangulation, inspect anchor precision/grounding and objective balance rather than the predictor.
- If triangulations are already wrong, later ICP and Chamfer improvements cannot repair semantic correspondence reliably.
- Confidence-weighted fusion can report low uncertainty when all views agree on the same wrong configuration; always inspect the spatial overlays.